# Decomposition Loss Weight Sweep — Local Windows

Same experiments as `colab_train_sample_eval2.ipynb` but runs locally.  
Requires the project to be at `c:\Users\ameli\Desktop\TezBaselines\MyCode`.

In [1]:
import os, sys
from pathlib import Path

REPO_PATH = r'c:\Users\ameli\Desktop\TezBaselines\MyCode'
assert Path(REPO_PATH).exists(), f'Not found: {REPO_PATH}'

os.chdir(REPO_PATH)
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'Memory  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Working directory: {os.getcwd()}')

PyTorch : 2.11.0+cu126
CUDA    : True
GPU     : NVIDIA GeForce RTX 4070
Memory  : 12.9 GB
Working directory: c:\Users\ameli\Desktop\TezBaselines\MyCode


In [2]:
import wandb, os
os.environ['WANDB_API_KEY'] = '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'

_api_key = os.environ.get('WANDB_API_KEY')
if _api_key:
    wandb.login(key=_api_key, relogin=False)
else:
    wandb.login()

print('wandb version:', wandb.__version__)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\ameli\_netrc
wandb: Currently logged in as: a-meliksahdemir (a-meliksahdemir-bo-azi-i-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb version: 0.27.0


In [ ]:
import torch, gc, importlib
from pathlib import Path

import train_with_mode as _twm
importlib.reload(_twm)
from train_with_mode import train

import evaluate_unified as _eu
importlib.reload(_eu)
from evaluate_unified import evaluate

DEVICE        = 'cuda' if torch.cuda.is_available() else 'cpu'
WANDB_PROJECT = 'diffusion-timeseries'
NUM_WORKERS   = 0     # must be 0 on Windows
SEED          = 42
BATCH_SIZE    = 64

NUM_EPOCHS          = 4000
LR                  = 1e-4
EVAL_METRICS_EVERY  = 200
N_METRIC_ITERATIONS = 3

# ── Experiment list ────────────────────────────────────────────────────────────
# 4 loss combinations × 2 model sizes = 8 experiments
# Large  : h256 / l8    Medium : h128 / l6

EXPERIMENTS = [

    dict(name='256_8large_fft_fixed',    group='largehiddendims',
         hidden_dim=256, num_layers=8,
         fft_weight=1.0, trend_weight=0.0, season_weight=0.0),

    dict(name='128_8large_trend_season', group='largehiddendims',
         hidden_dim=128, num_layers=8,
         fft_weight=1.0, trend_weight=0.0, season_weight=0.0),

    dict(name='384_8large_all_three',    group='largehiddendims',
         hidden_dim=384, num_layers=8,
         fft_weight=1.0, trend_weight=0.0, season_weight=0.0),

    dict(name='256_6layer_large',    group='largelayers',
         hidden_dim=256, num_layers=6,
         fft_weight=1.0, trend_weight=1.0, season_weight=0.0),

    dict(name='256_8layer_large',   group='largelayers',
         hidden_dim=256, num_layers=8,
         fft_weight=1.0, trend_weight=0.0, season_weight=1.0),
    
    dict(name='256_10layer_large',    group='largelayers',
        hidden_dim=256, num_layers=10,
        fft_weight=1.0, trend_weight=1.0, season_weight=1.0),

    dict(name='384_12layer_large',    group='largelayers',
        hidden_dim=384, num_layers=12,
        fft_weight=1.0, trend_weight=1.0, season_weight=1.0),
]


# ── Print plan ─────────────────────────────────────────────────────────────────
print(f'Device  : {DEVICE}')
print(f'Total   : {len(EXPERIMENTS)} experiments  |  epochs={NUM_EPOCHS}  eval_every={EVAL_METRICS_EVERY}  n_iter={N_METRIC_ITERATIONS}\n')
print(f"  {'#':<4} {'Name':<30} {'Group':<8} {'h':>5} {'l':>4} {'fft':>5} {'trend':>6} {'season':>7}")
print('  ' + '-'*68)
for i, exp in enumerate(EXPERIMENTS):
    print(f"  {i+1:<4} {exp['name']:<30} {exp['group']:<8}"
          f" {exp['hidden_dim']:>5} {exp['num_layers']:>4}"
          f" {exp['fft_weight']:>5.1f} {exp['trend_weight']:>6.1f} {exp['season_weight']:>7.1f}")
print()

# ── Run loop ───────────────────────────────────────────────────────────────────
for i, exp in enumerate(EXPERIMENTS):
    print(f"\n{'='*70}")
    print(f"  [{i+1}/{len(EXPERIMENTS)}]  {exp['name']}  (h={exp['hidden_dim']} l={exp['num_layers']})")
    print(f"{'='*70}\n")

    ckpt_dir  = f"output/ckpt_{exp['name']}"
    best_ckpt = f"{ckpt_dir}/best_model.pt"

    try:
        train(
            mode               = 'decomposition',
            device             = DEVICE,
            hidden_dim         = exp['hidden_dim'],
            num_layers         = exp['num_layers'],
            num_epochs         = NUM_EPOCHS,
            lr                 = LR,
            batch_size         = BATCH_SIZE,
            num_workers        = NUM_WORKERS,
            seed               = SEED,
            use_wandb          = True,
            wandb_project      = WANDB_PROJECT,
            wandb_run_name     = exp['name'],
            wandb_group        = exp['group'],
            eval_metrics       = True,
            eval_metrics_every = EVAL_METRICS_EVERY,
            n_metric_iterations= N_METRIC_ITERATIONS,
            img_pred_objective = 'pred_x0',
            img_loss_type      = 'l1',
            fft_weight         = exp['fft_weight'],
            trend_weight       = exp['trend_weight'],
            season_weight      = exp['season_weight'],
            checkpoint_dir     = ckpt_dir,
            finish_wandb       = False,
        )
    except Exception:
        import traceback
        print(f"\n!!! TRAINING FAILED: {exp['name']}")
        traceback.print_exc()
        try:
            import wandb
            if wandb.run is not None: wandb.finish()
        except Exception:
            pass
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        continue

    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    if Path(best_ckpt).exists():
        print(f"\n--- Post-training evaluation: {exp['name']} ---")
        try:
            evaluate(
                mode               = 'decomposition',
                checkpoint_path    = best_ckpt,
                device             = DEVICE,
                num_samples        = 256,
                n_metric_iterations= N_METRIC_ITERATIONS,
                compute_context_fid= True,
                use_wandb          = True,
                wandb_project      = WANDB_PROJECT,
                wandb_run_name     = exp['name'],
                wandb_group        = exp['group'],
                output_dir         = ckpt_dir,
                hidden_dim         = exp['hidden_dim'],
                num_layers         = exp['num_layers'],
            )
        except Exception:
            import traceback
            print(f"\n!!! EVALUATION FAILED: {exp['name']}")
            traceback.print_exc()
            try:
                import wandb
                if wandb.run is not None: wandb.finish()
            except Exception:
                pass
    else:
        print(f"   [skip eval] best_model.pt not found at {best_ckpt}")
        try:
            import wandb
            if wandb.run is not None: wandb.finish()
        except Exception:
            pass

    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print('\n' + '='*70)
print('  ALL EXPERIMENTS COMPLETE')
print('='*70)

Device  : cuda
Total   : 7 experiments  |  epochs=4000  eval_every=200  n_iter=3

  #    Name                           Group        h    l   fft  trend  season
  --------------------------------------------------------------------
  1    256_8large_fft                 largehiddendims   256    8   1.0    0.0     0.0
  2    128_8large_trend_season        largehiddendims   128    8   1.0    0.0     0.0
  3    384_8large_all_three           largehiddendims   384    8   1.0    0.0     0.0
  4    256_6layer_large               largelayers   256    6   1.0    1.0     0.0
  5    256_8layer_large               largelayers   256    8   1.0    0.0     1.0
  6    256_10layer_large              largelayers   256   10   1.0    1.0     1.0
  7    384_12layer_large              largelayers   384   12   1.0    1.0     1.0


  [1/7]  256_8large_fft  (h=256 l=8)

Global seed set to 42


epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
lr,▁████████████████████████
train_loss,█▄▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,█▄▁
best_epoch,21
best_val_loss,0.7931
epoch,25
lr,0.0001
train_loss,0.81508
val_loss,0.7931


TRANSFORMER DIFFUSION MODEL - DECOMPOSITION MODE
{'model': {'hidden_dim': 256, 'num_layers': 8, 'num_heads': 8, 'ff_dim': 1024, 'dropout': 0.1, 'input_channels': 6, 'sequence_length': 32, 'learnable_pos_enc': True}, 'diffusion': {'num_timesteps': 1000, 'beta_start': 0.0001, 'beta_end': 0.02, 'noise_schedule': 'cosine', 'variance_type': 'fixed_large', 'gamma': 1.0}, 'training': {'batch_size': 64, 'learning_rate': 0.0001, 'num_epochs': 4000, 'warmup_steps': 100, 'weight_decay': 0.0001, 'gradient_clip_val': 1.0, 'lr_scheduler_type': 'cosine', 'checkpoint_dir': 'output/ckpt_256_8large_fft', 'log_dir': 'c:\\Users\\ameli\\Desktop\\TezBaselines\\MyCode\\output\\logs', 'save_every_n_epochs': 250, 'validate_every_n_epochs': 10}, 'sampling': {'sampler_type': 'ddim', 'num_sampling_steps': 500, 'eta': 0.0, 'batch_size': 16, 'output_dir': 'c:\\Users\\ameli\\Desktop\\TezBaselines\\MyCode\\output\\generated_samples'}, 'data': {'data_path': 'c:\\Users\\ameli\\Desktop\\TezBaselines\\MyCode\\dataset\\st

KeyboardInterrupt: 

In [ ]:
import torch, gc
from pathlib import Path
from train_with_mode import train
from evaluate_unified import evaluate
import torch, gc, importlib
from pathlib import Path

import train_with_mode as _twm
importlib.reload(_twm)
from train_with_mode import train

import evaluate_unified as _eu
importlib.reload(_eu)
from evaluate_unified import evaluate

DEVICE        = 'cuda' if torch.cuda.is_available() else 'cpu'
WANDB_PROJECT = 'diffusion-timeseries'
NUM_WORKERS   = 0     # must be 0 on Windows
SEED          = 42
BATCH_SIZE    = 64

NUM_EPOCHS          = 2000
LR                  = 1e-4
EVAL_METRICS_EVERY  = 200
N_METRIC_ITERATIONS = 3

# ── Large config — dominant loss experiments ───────────────────────────────────
# Each experiment: one loss at 1.0, the other two at 0.5 or 0.25
# 3 × (one dominant) × 2 scales = 6 experiments, all h256/l8

DOMINANT_EXPERIMENTS = [

    # ── One loss dominant at 1.0, others at 0.5 ───────────────────────────────
    dict(name='large_fft1_rest05_local',    group='large_dominant_05',
         hidden_dim=256, num_layers=8,
         fft_weight=1.0, trend_weight=0.5, season_weight=0.5),

    dict(name='large_trend1_rest05_local',  group='large_dominant_05',
         hidden_dim=256, num_layers=8,
         fft_weight=0.5, trend_weight=1.0, season_weight=0.5),

    dict(name='large_season1_rest05_local', group='large_dominant_05',
         hidden_dim=256, num_layers=8,
         fft_weight=0.5, trend_weight=0.5, season_weight=1.0),

    # ── One loss dominant at 1.0, others at 0.25 ──────────────────────────────
    dict(name='large_fft1_rest025_local',    group='large_dominant_025',
         hidden_dim=256, num_layers=8,
         fft_weight=1.0, trend_weight=0.25, season_weight=0.25),

    dict(name='large_trend1_rest025_local',  group='large_dominant_025',
         hidden_dim=256, num_layers=8,
         fft_weight=0.25, trend_weight=1.0, season_weight=0.25),

    dict(name='large_season1_rest025_local', group='large_dominant_025',
         hidden_dim=256, num_layers=8,
         fft_weight=0.25, trend_weight=0.25, season_weight=1.0),
]

# ── Print plan ─────────────────────────────────────────────────────────────────
print(f'Total : {len(DOMINANT_EXPERIMENTS)} experiments  |  epochs={NUM_EPOCHS}  eval_every={EVAL_METRICS_EVERY}  n_iter={N_METRIC_ITERATIONS}\n')
print(f"  {'#':<4} {'Name':<30} {'Group':<22} {'fft':>5} {'trend':>6} {'season':>7}")
print('  ' + '-'*72)
for i, exp in enumerate(DOMINANT_EXPERIMENTS):
    print(f"  {i+1:<4} {exp['name']:<30} {exp['group']:<22}"
          f" {exp['fft_weight']:>5.2f} {exp['trend_weight']:>6.2f} {exp['season_weight']:>7.2f}")
print()

# ── Run loop ───────────────────────────────────────────────────────────────────
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

for i, exp in enumerate(DOMINANT_EXPERIMENTS):
    print(f"\n{'='*70}")
    print(f"  [{i+1}/{len(DOMINANT_EXPERIMENTS)}]  {exp['name']}  (group: {exp['group']})")
    print(f"{'='*70}\n")

    ckpt_dir  = f"output/ckpt_{exp['name']}"
    best_ckpt = f"{ckpt_dir}/best_model.pt"

    try:
        train(
            mode               = 'decomposition',
            device             = DEVICE,
            hidden_dim         = exp['hidden_dim'],
            num_layers         = exp['num_layers'],
            num_epochs         = NUM_EPOCHS,
            lr                 = LR,
            batch_size         = BATCH_SIZE,
            num_workers        = NUM_WORKERS,
            seed               = SEED,
            use_wandb          = True,
            wandb_project      = WANDB_PROJECT,
            wandb_run_name     = exp['name'],
            wandb_group        = exp['group'],
            eval_metrics       = True,
            eval_metrics_every = EVAL_METRICS_EVERY,
            n_metric_iterations= N_METRIC_ITERATIONS,
            img_pred_objective = 'pred_x0',
            img_loss_type      = 'l1',
            fft_weight         = exp['fft_weight'],
            trend_weight       = exp['trend_weight'],
            season_weight      = exp['season_weight'],
            checkpoint_dir     = ckpt_dir,
            finish_wandb       = False,
        )
    except Exception:
        import traceback
        print(f"\n!!! TRAINING FAILED: {exp['name']}")
        traceback.print_exc()
        try:
            import wandb
            if wandb.run is not None: wandb.finish()
        except Exception:
            pass
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        continue

    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    if Path(best_ckpt).exists():
        print(f"\n--- Post-training evaluation: {exp['name']} ---")
        try:
            evaluate(
                mode               = 'decomposition',
                checkpoint_path    = best_ckpt,
                device             = DEVICE,
                num_samples        = 256,
                n_metric_iterations= N_METRIC_ITERATIONS,
                compute_context_fid= True,
                use_wandb          = True,
                wandb_project      = WANDB_PROJECT,
                wandb_run_name     = exp['name'],
                wandb_group        = exp['group'],
                output_dir         = ckpt_dir,
                hidden_dim         = exp['hidden_dim'],
                num_layers         = exp['num_layers'],
            )
        except Exception:
            import traceback
            print(f"\n!!! EVALUATION FAILED: {exp['name']}")
            traceback.print_exc()
            try:
                import wandb
                if wandb.run is not None: wandb.finish()
            except Exception:
                pass
    else:
        print(f"   [skip eval] best_model.pt not found at {best_ckpt}")
        try:
            import wandb
            if wandb.run is not None: wandb.finish()
        except Exception:
            pass

    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print('\n' + '='*70)
print('  DOMINANT LOSS EXPERIMENTS COMPLETE')
print('='*70)

Total : 6 experiments  |  epochs=2000  eval_every=200  n_iter=3

  #    Name                           Group                    fft  trend  season
  ------------------------------------------------------------------------
  1    large_fft1_rest05_local        large_dominant_05       1.00   0.50    0.50
  2    large_trend1_rest05_local      large_dominant_05       0.50   1.00    0.50
  3    large_season1_rest05_local     large_dominant_05       0.50   0.50    1.00
  4    large_fft1_rest025_local       large_dominant_025      1.00   0.25    0.25
  5    large_trend1_rest025_local     large_dominant_025      0.25   1.00    0.25
  6    large_season1_rest025_local    large_dominant_025      0.25   0.25    1.00


  [1/6]  large_fft1_rest05_local  (group: large_dominant_05)

Global seed set to 42


TRANSFORMER DIFFUSION MODEL - DECOMPOSITION MODE
{'model': {'hidden_dim': 256, 'num_layers': 8, 'num_heads': 8, 'ff_dim': 1024, 'dropout': 0.1, 'input_channels': 6, 'sequence_length': 32, 'learnable_pos_enc': True}, 'diffusion': {'num_timesteps': 1000, 'beta_start': 0.0001, 'beta_end': 0.02, 'noise_schedule': 'cosine', 'variance_type': 'fixed_large', 'gamma': 1.0}, 'training': {'batch_size': 64, 'learning_rate': 0.0001, 'num_epochs': 2000, 'warmup_steps': 100, 'weight_decay': 0.0001, 'gradient_clip_val': 1.0, 'lr_scheduler_type': 'cosine', 'checkpoint_dir': 'output/ckpt_large_fft1_rest05_local', 'log_dir': 'c:\\Users\\ameli\\Desktop\\TezBaselines\\MyCode\\output\\logs', 'save_every_n_epochs': 250, 'validate_every_n_epochs': 10}, 'sampling': {'sampler_type': 'ddim', 'num_sampling_steps': 500, 'eta': 0.0, 'batch_size': 16, 'output_dir': 'c:\\Users\\ameli\\Desktop\\TezBaselines\\MyCode\\output\\generated_samples'}, 'data': {'data_path': 'c:\\Users\\ameli\\Desktop\\TezBaselines\\MyCode\\da

context_fid,█▃▄▅▃▃▂▃▁▁▃
correlational_score,▂█▂▃▁▅▂▂▂▁▁
disc_score,█▆▄▃▂▃▂▂▂▁▂
disc_score_std,▁▄█▆▁▂▁▄█▂▄
epoch,▁▁▁▁▁▂▂▂▂▂▂▂▂▃▃▃▃▃▃▃▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇███
fdds,▃▂█▃▇▃▁▁▂▁▂
lr,██████████▇▇▇▇▇▆▆▆▅▅▅▅▄▄▄▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁
pred_mae,█▂▂▂▂▁▂▂▁▁▁
pred_mae_std,▆▄▇█▁▂▁▄▅▁▂
test_acc,█▆▄▃▂▃▂▂▂▁▂
+3,...


   W&B run finished.

Evaluation complete! (decomposition mode)
Results saved to: output/ckpt_large_fft1_rest05_local

  [2/6]  large_trend1_rest05_local  (group: large_dominant_05)

Global seed set to 42


TRANSFORMER DIFFUSION MODEL - DECOMPOSITION MODE
{'model': {'hidden_dim': 256, 'num_layers': 8, 'num_heads': 8, 'ff_dim': 1024, 'dropout': 0.1, 'input_channels': 6, 'sequence_length': 32, 'learnable_pos_enc': True}, 'diffusion': {'num_timesteps': 1000, 'beta_start': 0.0001, 'beta_end': 0.02, 'noise_schedule': 'cosine', 'variance_type': 'fixed_large', 'gamma': 1.0}, 'training': {'batch_size': 64, 'learning_rate': 0.0001, 'num_epochs': 2000, 'warmup_steps': 100, 'weight_decay': 0.0001, 'gradient_clip_val': 1.0, 'lr_scheduler_type': 'cosine', 'checkpoint_dir': 'output/ckpt_large_trend1_rest05_local', 'log_dir': 'c:\\Users\\ameli\\Desktop\\TezBaselines\\MyCode\\output\\logs', 'save_every_n_epochs': 250, 'validate_every_n_epochs': 10}, 'sampling': {'sampler_type': 'ddim', 'num_sampling_steps': 500, 'eta': 0.0, 'batch_size': 16, 'output_dir': 'c:\\Users\\ameli\\Desktop\\TezBaselines\\MyCode\\output\\generated_samples'}, 'data': {'data_path': 'c:\\Users\\ameli\\Desktop\\TezBaselines\\MyCode\\

...

   W&B run finished.

Evaluation complete! (decomposition mode)
Results saved to: output/ckpt_large_season1_rest05_local

  [4/6]  large_fft1_rest025_local  (group: large_dominant_025)

Global seed set to 42


TRANSFORMER DIFFUSION MODEL - DECOMPOSITION MODE
{'model': {'hidden_dim': 256, 'num_layers': 8, 'num_heads': 8, 'ff_dim': 1024, 'dropout': 0.1, 'input_channels': 6, 'sequence_length': 32, 'learnable_pos_enc': True}, 'diffusion': {'num_timesteps': 1000, 'beta_start': 0.0001, 'beta_end': 0.02, 'noise_schedule': 'cosine', 'variance_type': 'fixed_large', 'gamma': 1.0}, 'training': {'batch_size': 64, 'learning_rate': 0.0001, 'num_epochs': 2000, 'warmup_steps': 100, 'weight_decay': 0.0001, 'gradient_clip_val': 1.0, 'lr_scheduler_type': 'cosine', 'checkpoint_dir': 'output/ckpt_large_fft1_rest025_local', 'log_dir': 'c:\\Users\\ameli\\Desktop\\TezBaselines\\MyCode\\output\\logs', 'save_every_n_epochs': 250, 'validate_every_n_epochs': 10}, 'sampling': {'sampler_type': 'ddim', 'num_sampling_steps': 500, 'eta': 0.0, 'batch_size': 16, 'output_dir': 'c:\\Users\\ameli\\Desktop\\TezBaselines\\MyCode\\output\\generated_samples'}, 'data': {'data_path': 'c:\\Users\\ameli\\Desktop\\TezBaselines\\MyCode\\d

context_fid,█▄▄▅▄▃▁▃▁▂▃
correlational_score,▅█▂▂▁▃▁▁▃▁▁
disc_score,█▅▄▂▃▂▃▂▂▁▂
disc_score_std,▁▂▄▃▁▄▁▇█▁▃
epoch,▁▁▁▁▁▂▂▂▂▂▂▂▂▃▃▄▄▄▄▄▄▄▄▄▄▅▅▆▆▆▆▆▇▇▇▇▇▇██
fdds,▁██▃▁█▇▁▁▁▇
lr,██████████▇▇▇▇▇▆▆▅▅▄▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁
pred_mae,█▃▁▂▂▁▂▂▁▁▂
pred_mae_std,██▂▇▅▁▄▇█▂▃
test_acc,█▅▄▂▃▂▃▂▂▁▂
+3,...


   W&B run finished.

Evaluation complete! (decomposition mode)
Results saved to: output/ckpt_large_fft1_rest025_local

  [5/6]  large_trend1_rest025_local  (group: large_dominant_025)

Global seed set to 42


TRANSFORMER DIFFUSION MODEL - DECOMPOSITION MODE
{'model': {'hidden_dim': 256, 'num_layers': 8, 'num_heads': 8, 'ff_dim': 1024, 'dropout': 0.1, 'input_channels': 6, 'sequence_length': 32, 'learnable_pos_enc': True}, 'diffusion': {'num_timesteps': 1000, 'beta_start': 0.0001, 'beta_end': 0.02, 'noise_schedule': 'cosine', 'variance_type': 'fixed_large', 'gamma': 1.0}, 'training': {'batch_size': 64, 'learning_rate': 0.0001, 'num_epochs': 2000, 'warmup_steps': 100, 'weight_decay': 0.0001, 'gradient_clip_val': 1.0, 'lr_scheduler_type': 'cosine', 'checkpoint_dir': 'output/ckpt_large_trend1_rest025_local', 'log_dir': 'c:\\Users\\ameli\\Desktop\\TezBaselines\\MyCode\\output\\logs', 'save_every_n_epochs': 250, 'validate_every_n_epochs': 10}, 'sampling': {'sampler_type': 'ddim', 'num_sampling_steps': 500, 'eta': 0.0, 'batch_size': 16, 'output_dir': 'c:\\Users\\ameli\\Desktop\\TezBaselines\\MyCode\\output\\generated_samples'}, 'data': {'data_path': 'c:\\Users\\ameli\\Desktop\\TezBaselines\\MyCode\